In [ ]:
import pylab as plt
from IPython.display import clear_output
import numpy as np
import asyncio
from pathlib import Path

# Start Jupyter in the repository root or in code/.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "code":
    PROJECT_DIR = PROJECT_DIR.parent
INPUT_DIR = PROJECT_DIR / "data" / "spe"
OUTPUT_DIR = PROJECT_DIR / "runs" / "reinforce" / "spe"
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"Start Jupyter from the repository root or code directory: {PROJECT_DIR}")

class TSP(object):

    def __init__(self, 
                 num_cities, 
                 data_cnt, 
                 alpha=2, 
                 beta=1,
                 learning_rate=0.002,
                 eps=0.1,
                ):
        '''
        Args:
            num_cities (int): 城市数目
            map_size (int, int): 地图尺寸（宽，高）
            alpha (float): 一个超参，值越大，越优先探索最近的点
            beta (float): 一个超参，值越大，越优先探索可能导向总距离最优的点
            learning_rate (float)： 学习率
            eps (float): 探索率，值越大，探索性越强，但越难收敛。 可看作变异率
        '''
        self.num_cities = num_cities
        self.data_cnt = data_cnt
        self.alpha = alpha
        self.beta = beta
        self.eps = eps
        self.learning_rate = learning_rate
        self.cities = self.get_coords(data_cnt)
        self.distances = self.get_dist_matrix()
        self.mean_distance = self.distances.mean()
        self.qualities = np.zeros([num_cities, num_cities])
        self.best_path = None
        self.best_path_length = np.inf


    def get_coords(self, data_cnt):
        coords = [[], []]
        with (INPUT_DIR / f'{data_cnt}.in').open('r') as f:
            for line in f:
                try:
                    x, y = line.strip().split()
                except:
                    continue
                coords[0].append(float(x))
                coords[1].append(float(y))
        coords = np.array(coords)
        return coords

    def get_dist_matrix(self):
        dist_matrix = np.zeros([self.num_cities, self.num_cities])
        for i in range(self.num_cities):
            for j in range(self.num_cities):
                if i == j:
                    continue
                xi, xj = self.cities[0, i], self.cities[0, j]
                yi, yj = self.cities[1, i], self.cities[1, j]
                dist_matrix[i, j] = np.sqrt((xi-xj)**2 + (yi-yj)**2)
        return dist_matrix

    def rollout(self, start_city_id=None):
        cities_visited = []
        action_probs = []
        if start_city_id is None:
            start_city_id = np.random.randint(self.num_cities)
        current_city_id = start_city_id
        cities_visited.append(current_city_id)
        while len(cities_visited) < self.num_cities:
            current_city_id, action_prob = self.choose_next_city(cities_visited)
            cities_visited.append(current_city_id)
            action_probs.append(action_prob)
        action_probs.append(1.0)

        path_length = self.calc_path_length(cities_visited)
        if path_length < self.best_path_length:
            self.best_path = cities_visited
            self.best_path_length = path_length
        rewards = self.calc_path_rewards(cities_visited, path_length)
        return cities_visited, action_probs, rewards

    def choose_next_city(self, cities_visited):
        current_city_id = cities_visited[-1]

        probabilities = np.exp(self.qualities[current_city_id])

        for city_visited in cities_visited:
            probabilities[city_visited] = 0

        probabilities = probabilities/probabilities.sum()
        
        if np.random.random() < self.eps:
            next_city_id = np.random.choice(range(len(probabilities)), p=probabilities)
        else:
            next_city_id = probabilities.argmax()

        if probabilities.argmax() == next_city_id:
            action_prob = probabilities[next_city_id]*self.eps + (1-self.eps)
        else:
            action_prob = probabilities[next_city_id]*self.eps
            
        return next_city_id, action_prob

    def calc_path_rewards(self, path, path_length):
        rewards = []
        for fr, to in zip(path[:-1], path[1:]):
            dist = self.distances[fr, to]
            reward = (self.mean_distance/path_length)**self.beta
            reward = reward*(self.mean_distance/dist)**self.alpha
            rewards.append(np.log(reward))
        return rewards

    def calc_path_length(self, path):
        path_length = 0
        for fr, to in zip(path[:-1], path[1:]):
            path_length += self.distances[fr, to]
        return path_length
    
    def calc_updates_for_one_rollout(self, path, action_probs, rewards):
        qualities = []
        for fr, to, reward, action_prob in zip(path[:-1], path[1:], rewards, action_probs):
            log_action_probability = np.log(action_prob)
            qualities.append(- reward*log_action_probability)
        return qualities

    def update(self, path, new_qualities):
        lr = self.learning_rate
        for fr, to, new_quality in zip(
            path[:-1], path[1:], new_qualities):
            self.qualities[fr, to] = (1-lr)*self.qualities[fr, to] + lr*new_quality
    
    async def train_for_one_rollout(self, start_city_id):
        path, action_probs, rewards = self.rollout(start_city_id=start_city_id)
        new_qualities = self.calc_updates_for_one_rollout(path, action_probs, rewards)
        self.update(path, new_qualities)

    async def train_for_one_epoch(self):
        tasks = [self.train_for_one_rollout(start_city_id) for start_city_id in range(self.num_cities)]
        await asyncio.gather(*tasks)

    async def train(self, num_epochs=1000, display=True):
        for epoch in range(1, num_epochs + 1):
            await self.train_for_one_epoch()
            if display:
                clear_output()
                print(f'epoch {epoch} done')
                print(f'best path length: {self.best_path_length}')

    def print_ans(self):
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        out_path = OUTPUT_DIR / f'new_{self.data_cnt}.out'
        ans_path = OUTPUT_DIR / f'new_{self.data_cnt}.ans'
        if out_path.exists() or ans_path.exists():
            raise FileExistsError(f"Choose unused output names in: {OUTPUT_DIR}")
        with out_path.open('x') as f:
            f.write(f'{self.best_path_length}\n')
        with ans_path.open('x') as f:
            for city in self.best_path:
                f.write(f'{city + 1} ')

num_cities = [100]
data_num = len(list(INPUT_DIR.glob("*.in")))

for num_city in num_cities:
    for i in range(1, data_num + 1):
        tsp = TSP(num_city, i)
        await tsp.train(1000, display=True)
        print(f'data {num_city}, id {i} done')
        tsp.print_ans()
